# Overshoot permafrost pipeline

Runs the calculations and figures for one model / forcing / simulation combination.

**How to use**
1. Edit `config/run_config.yaml` (created automatically on first run) to choose the run and options.
2. Run all cells (*Run > Run All Cells*). If you change the YAML later, restart the kernel and run all.
3. Figures are saved under `figures/<run tag>/` and shown at the bottom of this notebook.

Please do not edit this notebook in place, so that updates pulled from GitHub apply cleanly. Use *File > Save As* to make a personal copy if you want to customise it.

## 1. Setup

In [ ]:
from __future__ import annotations

import importlib
import inspect
import shutil
import sys
import time
import traceback
from pathlib import Path

from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    """Walk up from the working directory until the folder containing code/wiemip_io.py is found."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "code" / "wiemip_io.py").is_file():
            return candidate
    raise RuntimeError(f"Could not find the project root (code/wiemip_io.py) above {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "code") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "code"))

import wiemip_io
import plotting

print(f"Project root : {PROJECT_ROOT}")

## 2. Pipeline modules

CALC_NAMES = [
    "calc_permafrost_extent",   # first: writes the permafrost mask others may use
    "calc_carbon_series",
    "calc_extent_definitions",
    "calc_carbon_fluxes",
]
FIG_NAMES = [
    "fig_permafrost_extent",
    "fig_extent_definitions",
    "fig_carbon_pools_anomaly",
    "fig_carbon_components",
    "fig_carbon_fluxes",
    "fig_carbon_hysteresis",
    "fig_circumpolar_maps",
    "fig_domain_comparison",
    "fig_depth_bands",          # slow, gated by run_depth_bands in the config
]

MODULES: dict[str, object] = {}
IMPORT_ERRORS: dict[str, str] = {}


def load_modules(reload: bool = True):
    """Import (or reload) every pipeline module. One broken module won't stop the rest."""
    importlib.invalidate_caches()
    for name in CALC_NAMES + FIG_NAMES:
        try:
            if name in sys.modules and reload:
                MODULES[name] = importlib.reload(sys.modules[name])
            else:
                MODULES[name] = importlib.import_module(name)
            IMPORT_ERRORS.pop(name, None)
        except Exception as exc:
            MODULES.pop(name, None)
            IMPORT_ERRORS[name] = f"{type(exc).__name__}: {exc}"


# Reload shared helpers first so edits to them propagate to the scripts.
importlib.reload(wiemip_io)
importlib.reload(plotting)
load_modules()

print(f"Loaded : {len(MODULES)}/{len(CALC_NAMES) + len(FIG_NAMES)} modules")
for name, err in IMPORT_ERRORS.items():
    print(f"  IMPORT FAILED  {name}: {err}")

## 3. Configuration

Settings are read from `config/run_config.yaml`. On first run this file is created from `config/run_config.example.yaml`.
It is not tracked by git, so your settings are never overwritten when the repository is updated.

In [ ]:
import yaml

cfg_path = PROJECT_ROOT / "config" / "run_config.yaml"
example_path = PROJECT_ROOT / "config" / "run_config.example.yaml"

if not cfg_path.exists():
    shutil.copy(example_path, cfg_path)
    print(f"Created {cfg_path} from the example (default settings).")

cfg = yaml.safe_load(cfg_path.read_text())

required = {"model", "forcing", "simulation", "recompute", "run_depth_bands"}
missing = required - set(cfg)
if missing:
    raise KeyError(f"run_config.yaml is missing: {sorted(missing)}")

RUN = wiemip_io.Run(model=cfg["model"], forcing=cfg["forcing"], simulation=cfg["simulation"])
RECOMPUTE = bool(cfg["recompute"])             # True on a fresh run or after changing inputs
RUN_DEPTH_BANDS = bool(cfg["run_depth_bands"])  # slow (~10-15 min): scans 3D soil temperature

print(f"Config file : {cfg_path}")
print(f"Run         : {RUN.tag}")
print(f"recompute={RECOMPUTE}   run_depth_bands={RUN_DEPTH_BANDS}")

## 4. Steps to run

In [ ]:
# (module name, extra kwargs). `run` is always passed; `recompute` is passed
# only to functions whose signature accepts it.
CALC_STEPS = [
    ("calc_permafrost_extent",  {"method": "soilt"}),
    ("calc_carbon_series",      {}),
    ("calc_extent_definitions", {}),
    ("calc_carbon_fluxes",      {}),
]
FIG_STEPS = [
    ("fig_permafrost_extent",    {}),
    ("fig_extent_definitions",   {}),
    ("fig_carbon_pools_anomaly", {}),
    ("fig_carbon_components",    {}),
    ("fig_carbon_fluxes",        {}),
    ("fig_carbon_hysteresis",    {}),
    ("fig_circumpolar_maps",     {}),
    ("fig_domain_comparison",    {}),
    ("fig_depth_bands",          {"recompute": RUN_DEPTH_BANDS}),
]
SLOW_STEPS = {"fig_depth_bands": RUN_DEPTH_BANDS}   # name -> enabled?

## 5. Runner functions

In [ ]:
def run_one(name: str, extra: dict) -> str:
    """Run one module's main(run, ...) safely. Returns a status string."""
    if name in IMPORT_ERRORS:
        print(f"  IMPORT FAILED: {IMPORT_ERRORS[name]}")
        return "import-failed"

    if name in SLOW_STEPS and not SLOW_STEPS[name]:
        print("  skipped (slow step; set run_depth_bands: true in config/run_config.yaml)")
        return "skipped"

    func = getattr(MODULES[name], "main", None)
    if func is None:
        print("  no main() found")
        return "no-main"

    params = inspect.signature(func).parameters
    if "run" not in params:
        print(f"  NOT CONVERTED: main{inspect.signature(func)} does not take `run`.")
        print("  Skipped on purpose: running it would use its own hardcoded "
              "model/forcing/simulation.")
        return "not-converted"

    kwargs = {k: v for k, v in extra.items() if k in params}
    if "recompute" in params and "recompute" not in kwargs and name.startswith("calc_"):
        kwargs["recompute"] = RECOMPUTE

    try:
        func(RUN, **kwargs)
        return "ok"
    except Exception:
        traceback.print_exc(file=sys.stdout)   # stdout keeps output in order
        return "FAILED"


def run_steps(steps) -> dict[str, str]:
    status = {}
    for name, extra in steps:
        print(f"\n{'=' * 64}\n{name}\n{'=' * 64}")
        status[name] = run_one(name, extra)
    return status


def summarize(status: dict[str, str], title: str):
    print(f"\n{title}")
    for name, s in status.items():
        print(f"  [{s:>13}]  {name}")


def show_figures(only_new_since: float | None = None):
    """Display saved PNGs for this run. Pass a timestamp to show only newer ones."""
    pngs = sorted(
        (p for p in RUN.figure_dir.glob("*.png") if ".ipynb_checkpoints" not in p.parts),
        key=lambda p: p.stat().st_mtime,
    )
    if only_new_since is not None:
        pngs = [p for p in pngs if p.stat().st_mtime >= only_new_since]
    if not pngs:
        print(f"No figures in {RUN.figure_dir}")
    for p in pngs:
        print(p.name)
        display(Image(filename=str(p)))

## 6. Run the pipeline

Reloads the scripts (so any edits are picked up), runs all calculations, then all figures, and prints a status summary.

In [ ]:
load_modules()   # pick up any edits to the scripts
started = time.time()
print(f"Run: {RUN.tag}   |   recompute={RECOMPUTE}   |   depth bands={RUN_DEPTH_BANDS}")

calc_status = run_steps(CALC_STEPS)
fig_status = run_steps(FIG_STEPS)

print("\n" + "#" * 64)
summarize(calc_status, "CALCULATIONS")
summarize(fig_status, "FIGURES")
print(f"\nElapsed: {time.time() - started:.0f} s")

## 7. Figures

Shows the figures created in this run. To see everything saved for this run, call `show_figures()` with no argument.

In [ ]:
show_figures(only_new_since=started)